# Generación de Texto con GPT-OSS-20B vía Groq API

En este cuaderno usaremos **GPT-OSS-20B**, un modelo de pesos abiertos de **OpenAI** (agosto 2025, licencia Apache 2.0), a través de la **API de Groq**, sin descargarlo ni ejecutarlo localmente.

### ¿Qué es Groq?
- Groq es un servicio de inferencia en la nube que ejecuta modelos abiertos a **muy alta velocidad**.
- Usa hardware especializado (LPU - Language Processing Unit) optimizado para modelos de lenguaje.
- Ofrece una API compatible con el estándar OpenAI, lo que la hace fácil de usar.

### Ventaja vs modelo local
- **No necesitamos GPU** en Colab (la inferencia se hace en los servidores de Groq).
- La respuesta es **mucho más rápida** que ejecutar el modelo localmente (como hicimos con GPT-2).
- Solo necesitamos una **API Key** (Groq ofrece un plan gratuito con límites de uso).

> GPT-OSS es un modelo de **razonamiento**: antes de responder genera "tokens de razonamiento" internos que también cuentan dentro del límite de tokens. Por eso usaremos un límite de tokens generoso.

### ¿Cómo obtener la API Key de Groq?
1. Ir a [console.groq.com](https://console.groq.com)
2. Crear una cuenta (se puede usar Google o GitHub)
3. En el menú lateral, ir a **API Keys**
4. Click en **Create API Key**, darle un nombre y copiar la clave
5. En Colab, ir a **Secretos** (ícono de llave 🔑 en el panel izquierdo)
6. Agregar un secreto con nombre `GROQ_API_KEY` y pegar la clave (activar el acceso desde el cuaderno)

## Paso 1: Configurar la API Key

Cargamos la clave de API desde los secretos de Colab. Esto es más seguro que escribir la clave directamente en el código.

In [3]:
from google.colab import userdata
import os

# Obtener la API key desde los secretos de Colab
api_key = userdata.get('GROQ_API_KEY')

# Guardarla como variable de entorno (así el SDK de Groq la encuentra automáticamente)
os.environ['GROQ_API_KEY'] = api_key

# Verificar que se haya cargado correctamente (nunca imprimas la clave, ni siquiera parcialmente)
print("API Key cargada:", "Sí" if api_key else "No")

API Key cargada: Sí


## Paso 2: Instalar el SDK de Groq

La librería `groq` nos da acceso a la API de Groq desde Python.

In [4]:
!pip install -q -U groq

## Paso 3: Crear el cliente de Groq

El cliente es el objeto que nos permite comunicarnos con la API.
Al tener la variable de entorno `GROQ_API_KEY` configurada, el cliente la detecta automáticamente.

In [5]:
from groq import Groq

# Crear el cliente de Groq
# La API key se lee automáticamente de la variable de entorno GROQ_API_KEY
client = Groq()

print("Cliente de Groq creado exitosamente!")

Cliente de Groq creado exitosamente!


## Paso 4: Generar texto con GPT-OSS-20B

Usamos `chat.completions.create()` para enviar mensajes al modelo y recibir respuestas.
El formato es el mismo estándar de OpenAI:

- `model`: el ID del modelo a usar (en Groq, el ID es `openai/gpt-oss-20b`; también existe `openai/gpt-oss-120b`, más grande).
- `messages`: lista de mensajes con roles (`system`, `user`, `assistant`).
- `temperature`: creatividad de la respuesta (0.0 a 2.0).
- `max_completion_tokens`: longitud máxima de la respuesta, **incluyendo** los tokens de razonamiento (reemplaza al antiguo `max_tokens`).
- `reasoning_effort`: cuánto "piensa" el modelo antes de responder (`low`, `medium` o `high`). Con `low` responde más rápido y gasta menos tokens.

In [6]:
# Enviamos una pregunta al modelo GPT-OSS-20B a través de Groq
respuesta = client.chat.completions.create(
    model="openai/gpt-oss-20b",  # Modelo GPT-OSS-20B de OpenAI hospedado en Groq
    messages=[
        {"role": "system", "content": "Eres un asistente experto y claro. Responde en español."},
        {"role": "user", "content": "Explica qué es la inteligencia artificial en 3 oraciones simples."}
    ],
    temperature=0.7,
    max_completion_tokens=1024,
    reasoning_effort="low",
)

# Extraemos el texto de la respuesta
texto = respuesta.choices[0].message.content

print("Pregunta: Explica qué es la inteligencia artificial en 3 oraciones simples.")
print("\nRespuesta del modelo:")
print(texto)

# Podemos ver estadísticas de uso
print(f"\n--- Estadísticas ---")
print(f"Tokens del prompt: {respuesta.usage.prompt_tokens}")
print(f"Tokens generados (incluye razonamiento): {respuesta.usage.completion_tokens}")
print(f"Tokens totales: {respuesta.usage.total_tokens}")

Pregunta: Explica qué es la inteligencia artificial en 3 oraciones simples.

Respuesta del modelo:
La inteligencia artificial (IA) es una rama de la informática que crea sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como aprender, razonar o tomar decisiones. Estas máquinas usan algoritmos y datos para reconocer patrones, mejorar su rendimiento con el tiempo y adaptarse a nuevas situaciones. En esencia, la IA permite que las computadoras imiten y, en algunos casos, superen la capacidad de procesamiento cognitivo humano.

--- Estadísticas ---
Tokens del prompt: 102
Tokens generados (incluye razonamiento): 101
Tokens totales: 203


## Paso 5: Función reutilizable

Creamos una función que encapsula la llamada a la API para poder reutilizarla fácilmente.

In [7]:
def generar_texto(prompt, max_tokens=1024, temperatura=0.7, esfuerzo="low"):
    """
    Genera texto usando GPT-OSS-20B a través de la API de Groq.

    Args:
        prompt (str): La pregunta o instrucción para el modelo.
        max_tokens (int): Máximo de tokens a generar (incluye los de razonamiento).
        temperatura (float): Creatividad del modelo (0.0 a 2.0).
        esfuerzo (str): Esfuerzo de razonamiento: "low", "medium" o "high".

    Returns:
        str: El texto generado por el modelo.
    """
    try:
        respuesta = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": "Eres un asistente experto y claro. Responde en español."},
                {"role": "user", "content": prompt}
            ],
            temperature=temperatura,
            max_completion_tokens=max_tokens,
            reasoning_effort=esfuerzo,
        )
        return (respuesta.choices[0].message.content or "").strip()

    except Exception as e:
        return f"Error en la generación: {str(e)}"

## Paso 6: Ejemplos prácticos

Probemos la función con diferentes tipos de prompts.

In [8]:
# Ejemplo 1: Pregunta educativa
respuesta1 = generar_texto("¿Cuáles son las 3 leyes de la robótica de Isaac Asimov?")
print("Ejemplo 1 - Pregunta educativa:")
print(respuesta1)
print("\n" + "="*70 + "\n")

# Ejemplo 2: Tarea creativa con temperatura alta
respuesta2 = generar_texto(
    "Escribe un párrafo corto sobre el futuro de la tecnología.",
    max_tokens=1024,
    temperatura=0.9
)
print("Ejemplo 2 - Tarea creativa (temperatura=0.9):")
print(respuesta2)

Ejemplo 1 - Pregunta educativa:
Las **tres leyes de la robótica** de Isaac Asimov son las siguientes:

1. **Primera Ley** – *Un robot no debe dañar a un ser humano ni, por inacción, permitir que un ser humano sufra daño.*  
2. **Segunda Ley** – *Un robot debe obedecer las órdenes que le dé un ser humano, salvo que esas órdenes entren en conflicto con la Primera Ley.*  
3. **Tercera Ley** – *Un robot debe proteger su propia existencia, siempre que dicha protección no entre en conflicto con la Primera o la Segunda Ley.*


Ejemplo 2 - Tarea creativa (temperatura=0.9):
El futuro de la tecnología se perfila como una convergencia de inteligencia artificial, biotecnología y redes cuánticas que transformará la forma en que vivimos, trabajamos y nos relacionamos. Los dispositivos inteligentes, integrados en la propia arquitectura de la ciudad, anticiparán nuestras necesidades y optimizarán recursos, mientras que la bioinformática permitirá personalizar tratamientos médicos al nivel genético. La